In [22]:
# EXTRAÇÃO DE DADOS

In [23]:
import requests
import pandas as pd

In [24]:
page = 0
items_per_page = 100

all_data = []

In [25]:
while True:
    url = f'https://api.obrasgov.gestao.gov.br/obrasgov/api/projeto-investimento?uf=DF&pagina={page}&tamanhoDaPagina={items_per_page}'
    raw_data = requests.get(url).json()
    data_content = raw_data['content']

    all_data.extend(data_content)

    current_page_elements_count = int(raw_data['numberOfElements'])

    print(f'current_page: {page} | elements_count: {current_page_elements_count}')

    page = page + 1

    if current_page_elements_count == 0:
        break

current_page: 0 | elements_count: 100
current_page: 1 | elements_count: 100
current_page: 2 | elements_count: 100
current_page: 3 | elements_count: 100
current_page: 4 | elements_count: 100
current_page: 5 | elements_count: 100
current_page: 6 | elements_count: 100
current_page: 7 | elements_count: 100
current_page: 8 | elements_count: 34
current_page: 9 | elements_count: 0


In [8]:
df = pd.DataFrame(all_data)


# cria uma visão "CSV-like": objetos viram string, como o to_csv faria
view = df.copy()
obj_cols = view.select_dtypes(include=["object"]).columns
view[obj_cols] = view[obj_cols].astype(str)

# mesma lógica do drop_duplicates do CSV (todas as colunas)
mask = ~view.duplicated(keep="first")
df = df.loc[mask].reset_index(drop=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 706 entries, 0 to 705
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   idUnico                             706 non-null    object
 1   nome                                706 non-null    object
 2   cep                                 344 non-null    object
 3   endereco                            380 non-null    object
 4   descricao                           706 non-null    object
 5   funcaoSocial                        706 non-null    object
 6   metaGlobal                          706 non-null    object
 7   dataInicialPrevista                 705 non-null    object
 8   dataFinalPrevista                   705 non-null    object
 9   dataInicialEfetiva                  25 non-null     object
 10  dataFinalEfetiva                    5 non-null      object
 11  dataCadastro                        706 non-null    object

In [9]:
df.shape

(706, 31)

In [10]:
df = df.convert_dtypes(
    infer_objects=True,
    convert_string=True,
    convert_integer=True,
    convert_boolean=True,
    convert_floating=True,
)

In [11]:
df.describe()

,idUnico,nome,cep,endereco,descricao,funcaoSocial,metaGlobal,dataInicialPrevista,dataFinalPrevista,dataInicialEfetiva,...,observacoesPertinentes,isModeladaPorBim,dataSituacao,tomadores,executores,repassadores,eixos,tipos,subTipos,fontesDeRecurso
count,706,706,344,380,706,706,706,705,705,25,...,89,497,706,706,706,706,706,706,706,706
unique,658,622,91,240,600,485,453,392,440,15,...,4,2,356,51,77,50,16,47,78,500
top,34841.53-39,CONSTRUÇÃO DE UNIDADE BÁSICA DE SAÚDE,1,,CONSTRUÇÃO DE UNIDADE BÁSICA DE SAÚDE,Segurança Pública,Escola de Educação Infantil Tipo B,2025-06-01,2027-06-01,2018-07-09,...,Informações Obras Fundo Nacional de Desenvolvi...,False,2025-07-25,[],[{'nome': 'FUNDO NACIONAL DE DESENVOLVIMENTO D...,[],"[{'id': 1, 'descricao': 'Administrativo'}]","[{'id': 5, 'descricao': 'Administrativo', 'idE...","[{'id': 59, 'descricao': 'Obras em Imóveis de ...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
freq,2,9,87,61,9,39,47,32,32,7,...,86,467,74,395,87,316,293,112,152,100


In [12]:
import json, ast
import pandas as pd

def parse_list_of_dicts_cell(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not (s.startswith('[') and s.endswith(']')):
        return [] 
    try:
        v = json.loads(s)
        return v if isinstance(v, list) else []
    except Exception:
        pass
    try:
        v = ast.literal_eval(s)
        return v if isinstance(v, list) else []
    except Exception:
        return []

In [13]:
colunas = [
    'tomadores',
    'executores',
    'repassadores',
    'eixos',
    'subTipos',
    'fontesDeRecurso'
]

In [14]:
def is_struct(x):
    return isinstance(x, (list, dict, set, tuple))

# identifica colunas com listas/dicts
cols_struct = [c for c in df.columns if df[c].apply(is_struct).any()]

# converte para string
for c in cols_struct:
    df[c] = df[c].astype(str)

In [15]:
import pandas as pd

df_out = df.copy()

for col_raw in colunas:
    df_exploded = df_out.copy()
    df_exploded["__items"] = df_exploded[col_raw].apply(parse_list_of_dicts_cell)

    df_exploded = df_exploded.explode("__items", ignore_index=True)

    norm = pd.json_normalize(df_exploded["__items"]).add_prefix(f"{col_raw}.")

    overlap = df_exploded.columns.intersection(norm.columns)
    if len(overlap) > 0:
        df_exploded = df_exploded.drop(columns=list(overlap))

    df_out = (
        df_exploded
        .drop(columns="__items")
        .reset_index(drop=True)
        .join(norm)
    )

    col_num = f"{col_raw}.valorInvestimentoPrevisto"
    if col_num in df_out.columns:
        s = df_out[col_num].astype("string")
        df_out[col_num] = pd.to_numeric(s, errors="coerce")

df_out = df_out.convert_dtypes()


In [16]:

df_out

,idUnico,nome,cep,endereco,descricao,funcaoSocial,metaGlobal,dataInicialPrevista,dataFinalPrevista,dataInicialEfetiva,...,executores.codigo,repassadores.nome,repassadores.codigo,eixos.id,eixos.descricao,subTipos.id,subTipos.descricao,subTipos.idTipo,fontesDeRecurso.origem,fontesDeRecurso.valorInvestimentoPrevisto
0,50379.53-54,DL - 304/2024 - Contratação de instituição par...,<NA>,<NA>,Contratação de instituição para execução de se...,Ampliação da capacidade de trafego visando a m...,Projetos Básicos e Executivos de Engenharia,2024-12-20,2027-12-05,<NA>,...,54844,<NA>,<NA>,3,Econômico,4,Acessos Terrestres,25,Federal,44463443.0
1,42724.53-27,Escola Classe Crixá São Sebastião,<NA>,<NA>,"Construção de Escola em Tempo Integral, Escola...",A construção da nova escola beneficiará 977 es...,"Construção de Escola em Tempo Integral, Escola...",2024-09-02,2028-09-02,<NA>,...,394676000107,FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCAÇÃO,253,4,Social,84,Educação,46,Federal,12319519.51
2,19970.53-78,Reajuste do Contrato 45/2021 - Contrução do Ce...,70.602-600,"SAIS Área Especial 3, Setor Policial Sul",Reajuste do Contrato 45/2021 - Construção do C...,Contribuir para a melhor formação dos bombeiro...,Construção de um novo centro de formação e de ...,2021-09-14,2024-08-28,<NA>,...,8977914000119,CORPO DE BOMBEIROS MILITAR DO DISTRITO FEDERAL,8977914000119,1,Administrativo,59,Obras em Imóveis de Uso Público,1,Federal,1177429.91
3,24797.53-15,Implantação de Passarelas nas Estradas Parque ...,<NA>,<NA>,Implantação de passarelas de estrutura mista n...,"Pedestres, no geral, demanda das ocupações lin...",Implantação de passarelas de estrutura mista n...,2023-08-30,2028-08-30,<NA>,...,70532000103,MINISTÉRIO DAS CIDADES,308798,3,Econômico,57,Obra de Arte Especial,24,Federal,10800000.0
4,24822.53-70,"obra de construção da Cabine de Medição, loca...",<NA>,<NA>,"obra de construção da Cabine de Medição, loca...",A demanda de carga elétrica do Campus Darcy Ri...,A demanda de carga elétrica do Campus Darcy Ri...,2023-09-14,2024-03-14,<NA>,...,26271,FUNDACAO UNIVERSIDADE DE BRASILIA,26271,3,Econômico,95,Subestação,31,Federal,928139.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1093,95925.53-27,REFORMA DO AUDITÓRIO DOIS CANDANGOS,<NA>,<NA>,REFORMA DO AUDITÓRIO DOIS CANDANGOS,REFORMA DO AUDITÓRIO DOIS CANDANGOS,REFORMA DO AUDITÓRIO DOIS CANDANGOS,2025-12-31,2026-10-31,<NA>,...,26271,FUNDACAO UNIVERSIDADE DE BRASILIA,26271,1,Administrativo,59,Obras em Imóveis de Uso Público,8,Federal,1656000.0
1094,94536.53-79,202411000093 Construção / Usina Fotovoltaica -...,<NA>,<NA>,202411000093 Construção / Usina Fotovoltaica -...,Função Social - Implantação de Usina Fotovolta...,Implantação de Usina Fotovoltaica.,2025-09-09,2026-03-31,<NA>,...,94,<NA>,<NA>,2,Militar,85,Estradas Vicinais,10,Federal,2749386.7
1095,28193.53-97,Implantação de passarelas em estrutura mista n...,<NA>,<NA>,Implantação de passarela de estrutura mista na...,População do Distrito Federal que utilizam as ...,Implantação de passarela de estrutura mista na...,2023-11-27,2027-11-27,<NA>,...,70532000103,MINISTÉRIO DAS CIDADES,308798,3,Econômico,57,Obra de Arte Especial,24,Federal,3120000.0
1096,74581.53-33,PAC 2 - Cobertura de Quadra Escolar 063,<NA>,<NA>,PAC 2 - Cobertura de Quadra Escolar 063 Tipo d...,PAC 2 - Cobertura de Quadra Escolar 063,Cobertura de Quadra Escolar Pequena,2013-12-06,2014-11-01,<NA>,...,253,<NA>,<NA>,3,Econômico,36,"Dragagem, Derrocamento e Associações",22,Federal,0.01


In [17]:
df_out = df_out.drop_duplicates()
df_out.reset_index(inplace=True)
df_out.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 839 entries, 0 to 838
Data columns (total 45 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   index                                      839 non-null    int64  
 1   idUnico                                    839 non-null    string 
 2   nome                                       839 non-null    string 
 3   cep                                        428 non-null    string 
 4   endereco                                   478 non-null    string 
 5   descricao                                  839 non-null    string 
 6   funcaoSocial                               839 non-null    string 
 7   metaGlobal                                 839 non-null    string 
 8   dataInicialPrevista                        838 non-null    string 
 9   dataFinalPrevista                          838 non-null    string 
 10  dataInicialEfetiva        

In [18]:
df_out.to_csv('resultado_final.csv', sep=';', encoding='utf-8-sig')

In [19]:
# CONEXÃO COM O BANCO DE DADOS

In [26]:
import os
import psycopg2
import psycopg2.extras
from sqlalchemy import create_engine

In [27]:
from sqlalchemy.engine import URL
url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("POSTGRES_HOST"),
    port=os.getenv("POSTGRES_PORT", "5432"),
    database=os.getenv("POSTGRES_DB"),
)
engine = create_engine(url, future=True)

with engine.begin() as sa_conn:  
    df_out.to_sql(
        name="projeto_investimento",
        con=sa_conn,
        schema="public",       
        if_exists="append",    
        index=False,
        chunksize=10_000,
        method="multi",        
    )